<a href="https://colab.research.google.com/github/Khushi-Kumari947/Cyber-security-chatbot/blob/main/notebooks/Fine_tune_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd

# Load your 76k Q&A pairs
df = pd.read_excel('/content/final_Training.xlsx')



In [ ]:
print(df.columns)

Index(['Question', 'Answer'], dtype='object')


In [ ]:

# Prepare training data: Question -> Answer pairs
train_examples = [
    InputExample(texts=[str(row['Question']), str(row['Answer'])])
    for _, row in df.iterrows()
]


In [ ]:
# Model: all-MiniLM-L6-v2 (80MB, fast, 384-dim)
model = SentenceTransformer('all-MiniLM-L6-v2')



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# DataLoader & Loss
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Fine-tune for 1 epoch (Domain Adaptation)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    show_progress_bar=True,
    output_path='cyber_model_tuned'
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,1.024629
1000,0.036372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load your new model
model = SentenceTransformer('./cyber_model_tuned')

# Test a cybersecurity specific term
query = "What is a zero-day vulnerability?"
# Pick a random answer from your dataset that is relevant
doc_1 = "A zero-day vulnerability is a software security flaw that is known to the software vendor but has no patch available."
doc_2 = "How to bake a chocolate cake."

emb1 = model.encode(query)
emb2 = model.encode(doc_1)
emb3 = model.encode(doc_2)

print(f"Similarity (Cyber): {util.cos_sim(emb1, emb2).item():.4f}")
print(f"Similarity (Random): {util.cos_sim(emb1, emb3).item():.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity (Cyber): 0.9125
Similarity (Random): -0.0201


In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.2 MB/s eta 0:00:00


In [ ]:
import boto3
import os
from google.colab import userdata

def upload_folder_to_s3(local_path, s3_prefix, bucket_name):
    # Retrieve keys securely from Colab Secrets
    access_key = userdata.get('AWS_ACCESS_KEY')
    secret_key = userdata.get('AWS_SECRET_KEY')

    s3_client = boto3.client(
        's3',
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name='us-east-1' # Change to your bucket's region
    )

    for root, dirs, files in os.walk(local_path):
        for file in files:
            local_file = os.path.join(root, file)
            relative_path = os.path.relpath(local_file, local_path)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")

            print(f"Uploading {local_file}...")
            s3_client.upload_file(local_file, bucket_name, s3_key)

# Usage
# Make sure the folder name matches exactly what is in your Colab file explorer
upload_folder_to_s3('/content/cyber_model_tuned', 'cyber_model_tuned', 'chatbot-assests')

Uploading /content/cyber_model_tuned/model.safetensors...
Uploading /content/cyber_model_tuned/sentence_bert_config.json...
Uploading /content/cyber_model_tuned/config_sentence_transformers.json...
Uploading /content/cyber_model_tuned/config.json...
Uploading /content/cyber_model_tuned/tokenizer_config.json...
Uploading /content/cyber_model_tuned/tokenizer.json...
Uploading /content/cyber_model_tuned/modules.json...
Uploading /content/cyber_model_tuned/README.md...
Uploading /content/cyber_model_tuned/1_Pooling/config.json...


In [ ]:
from google.colab import userdata
import boto3

# 1. Get keys again from Colab Secrets
access_key = userdata.get('AWS_ACCESS_KEY')
secret_key = userdata.get('AWS_SECRET_KEY')

# 2. Define the client globally
s3_client = boto3.client(
    's3',
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='us-east-1' # Make sure this matches your bucket region!
)

# 3. Now run your verification code
response = s3_client.list_objects_v2(Bucket='chatbot-assests', Prefix='cyber_model_tuned')

if 'Contents' in response:
    for obj in response.get('Contents', []):
        size_mb = obj['Size'] / (1024**2)
        print(f"✅ Found in S3: {obj['Key']} ({size_mb:.2f} MB)")
else:
    print("❌ No files found. Check your prefix or bucket name.")

✅ Found in S3: cyber_model_tuned/1_Pooling/config.json (0.00 MB)
✅ Found in S3: cyber_model_tuned/README.md (0.02 MB)
✅ Found in S3: cyber_model_tuned/config.json (0.00 MB)
✅ Found in S3: cyber_model_tuned/config_sentence_transformers.json (0.00 MB)
✅ Found in S3: cyber_model_tuned/model.safetensors (86.65 MB)
✅ Found in S3: cyber_model_tuned/modules.json (0.00 MB)
✅ Found in S3: cyber_model_tuned/sentence_bert_config.json (0.00 MB)
✅ Found in S3: cyber_model_tuned/tokenizer.json (0.68 MB)
✅ Found in S3: cyber_model_tuned/tokenizer_config.json (0.00 MB)
